# Predicting Assessment Scores

In [4]:
import os, shutil, warnings
import numpy as np
import pandas as pd
import kagglehub

# Visualization
import matplotlib.pyplot as plt

# Preprocessing, pipelines, validation, baselines
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

warnings.filterwarnings("ignore")


In [5]:
DATA_DIR = './data'

if not os.path.exists(DATA_DIR) or not os.listdir(DATA_DIR):
    print("Downloading dataset via kagglehub...")
    cache_path = kagglehub.competition_download('predicting-assessment-scores-hw1-ucu-ai-2026')
    print(f"Files cached at: {cache_path}")
    
    print(f"Copying dataset to {DATA_DIR}")
    shutil.copytree(cache_path, DATA_DIR, dirs_exist_ok=True)
    print("Data ready!")
else:
    print(f"Data already exists in {DATA_DIR}.")

print("Available files:", os.listdir(DATA_DIR))

Data already exists in ./data.
Available files: ['train.csv', 'test.csv', 'README.md', 'data_dictionary.csv', 'sample_submission.csv', 'unlabeled.csv']


In [6]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
train_df.head()

Train shape: (441000, 17)
Test shape:  (189000, 16)


,record_id,learner_age,gender_group,program_track,study_time,attendance_rate,home_internet,sleep_duration,rest_quality,learning_routine,campus_resources,assessment_level,study_minutes,engagement_index,intake_marker,registration_group,assessment_score
0,1572288,18.0,other,b.com,0.78,97.8,no,5.15,poor,mixed,low,moderate,43.9,70.74,0.743351,G06,48.5
1,1065516,21.0,other,b.tech,1.65,77.3,yes,6.41,poor,mixed,high,moderate,99.0,55.89,0.971166,G08,50.0
2,1480349,20.0,other,diploma,3.29,61.2,no,8.77,poor,self-study,low,easy,200.5,53.53,0.764103,G11,53.0
3,1128337,24.0,male,diploma,5.33,66.9,no,6.42,good,coaching,medium,moderate,332.0,67.67,0.865821,G10,89.9
4,1794604,23.0,female,bca,6.68,69.3,no,5.63,average,self-study,medium,easy,410.7,75.14,0.223452,G02,91.8


In [7]:
RANDOM_SEED = 42
N_SPLITS = 5

cv_splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

In [8]:
def evaluate_model(model, X, y, cv):
    """
    Evaluates a scikit-learn compatible model or pipeline using cross-validation.
    
    Parameters:
    - model: scikit-learn estimator or Pipeline
    - X: pandas DataFrame or numpy array of features
    - y: pandas Series or numpy array of the target variable
    - cv: scikit-learn cross-validation splitter (e.g., KFold)
    
    Returns:
    - dict containing OOF predictions and computed RMSE metrics
    """
    oof_predictions = np.zeros(len(X))
    fold_rmses = []

    X_arr = X.values if hasattr(X, 'values') else X
    y_arr = y.values if hasattr(y, 'values') else y

    print(f"Evaluating {model.__class__.__name__}...")

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_arr, y_arr)):
        X_train, X_val = X_arr[train_idx], X_arr[val_idx]
        y_train, y_val = y_arr[train_idx], y_arr[val_idx]
        
        fold_model = clone(model)

        fold_model.fit(X_train, y_train)

        val_preds = fold_model.predict(X_val)

        oof_predictions[val_idx] = val_preds

        fold_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
        fold_rmses.append(fold_rmse)
        print(f"  Fold {fold + 1} RMSE: {fold_rmse:.4f}")

    mean_rmse = np.mean(fold_rmses)
    std_rmse = np.std(fold_rmses)
    overall_rmse = np.sqrt(mean_squared_error(y_arr, oof_predictions))

    print("-" * 40)
    print(f"Mean Fold RMSE:   {mean_rmse:.4f} ± {std_rmse:.4f}")
    print(f"Overall OOF RMSE: {overall_rmse:.4f}")
    print("-" * 40)

    return {
        'oof_predictions': oof_predictions,
        'fold_rmses': fold_rmses,
        'mean_rmse': mean_rmse,
        'std_rmse': std_rmse,
        'overall_rmse': overall_rmse
    }